In [ ]:
include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )

base = "../data/results/";

In [ ]:
# Population sizes.
Nlist = [50,100,500,1000]
NN = length( Nlist )

# Case title.
μ = 1.0

# Temporal variables.
T = 100_000
tlist = 0:5:T
Nt = length( tlist )

# Adapatable time-step length.
smin = -3;  smax = 1;  δs = 0.25
slist = round.( 10.0.^(smin:δs:smax), digits=6 )  # NON-DIMENSIONAL.
Ns = length( slist )
println( "Running for $(Ns) different values of s0." )

In [ ]:
# Parameter variables.
nondimlist = [Nondim(; s=s ) for s ∈ slist]

# Data folder name.
folderdata = [[findfolder( N, μ, nondim; base=base ) for nondim ∈ nondimlist] for N ∈ Nlist];

In [ ]:
# Import composite state variable.
φdatadata = [[round.( defInt, readdlm( folder*"state-composition_T-$(T)_M-1.txt" ) )
    for folder ∈ folderlist] for folderlist ∈ folderdata]

# Extract number of active ants.
Adatalist = [[[sum( φdata[t,:] .== 0 ) for t ∈ 1:Nt]
    for φdata ∈ φdatalist] for (N, φdatalist) ∈ zip( Nlist, φdatadata )];

In [ ]:
function rankedprob(xlist::AbstractArray{defInt})::Vector{defInt}
    # Unpack the state variables as vectors.
    H = Dict{Vector{defInt},defInt}()

    # Iterate through the activity level combinations.
    for state ∈ eachrow( xlist )
        H[state] = get( H, state, 0 ) + 1
    end

    # Return the sorted frequency.
    return sort( collect( values( H ) ) )[end:-1:1]
end

# Initialize distribution count variable.
Rdata = [[rankedprob( Adatalist[i][j] ) for j ∈ 1:Ns] for i ∈ 1:NN]
rdata = [[R./sum( R ) for R ∈ Rlist] for Rlist ∈ Rdata];

In [ ]:
i = NN;  j = Ns-1

# Plot the activity burst.
plt = plot(; size=(300,200), dpi=100 )

plot!( plt, tlist, Adatalist[i][j]./Nlist[i]; color=:black, lw=2, label="" )

plot!( plt; xlims=(0,T/10), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1.0), ylabel="proportion of\nants active, "*L"\mathcal{A}" )

In [ ]:
# Compute the order parameter.
cut = 1000
ςdatalist = [[vec( readdlm( folderdata[i][j]*"phase-order_T-$(T).txt" ) )
    for j ∈ 1:Ns] for i ∈ 1:NN]

# Compute the mean and standard deviation of the order.
μςdata = hcat( [[mean( ςlist ) for ςlist ∈ ςdata] for ςdata ∈ ςdatalist]... )
σςdata = hcat( [[std(  ςlist ) for ςlist ∈ ςdata] for ςdata ∈ ςdatalist]... );

In [ ]:
i = 10
clist = [:black :cornflowerblue :indianred :mediumpurple :olivedrab]
llist = hcat( [latexstring( "N=$(N̂)" ) for N̂ ∈ Nlist]... )

# Plot the ranked probability distribution (at critical speed).
plt = plot(; size=(400,300), dpi=100, legend=:topleft )
plot!( plt; left_margin=5pt, bottom_margin=0pt, right_margin=10pt )

# Plot the temperature-dependent phase transition.
plot!( plt, slist, μςdata; ribbon=σςdata, color=clist, fillalpha=1/4, lw=2, marker=:circ, label=llist )

plot!( plt; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="speed constant, "*L"s_0", ylabel="mean phase coherence, "*L"⟨ς_χ⟩", legend=:topleft )

# Plot the probability of active state.
plot!( plt; inset=(1, bbox( 0.625, 0.56, 0.3, 0.3 )) )

for j ∈ NN:-1:1
    N̂ = length( rdata[j][i] )
    scatter!( plt[2], (1:N̂), rdata[j][i]; color=clist[j], alpha=1,
        markersize=3, markerstrokewidth=0 )
end

xmin = floor( log10( minimum( 1.0./Nlist ) ) )
xmax = ceil( log10( maximum( Nlist ) ) )
plot!( plt[2], 10.0.^[xmin,xmax], 10.0.^[-xmin,-xmax]; color=:black, linestyle=:dot, lw=2 )

ymin = -5;  ymax = 0
plot!( plt[2]; xlims=10.0.^[0,xmax], xticks=10.0.^[0,3], xscale=:log10 )
plot!( plt[2]; ylims=10.0.^[ymin,ymax], yticks=10.0.^[-5,0], yscale=:log10 )
plot!( plt[2]; framestyle=:box, legend=false )

annotate!( plt[2], 10.0^1.5, 10.0^-6.25, text( L"r(\mathcal{A})", :center, 10 ) )
annotate!( plt[2], 10.0^-0.65, 10.0^-2.5, text( L"P(\mathcal{A})", :center, 10; rotation=90 ) )

# saveplot( plt, figurefolder*"phase-coherence-spd_N-$(Nlist[1])-$(Nlist[end])"; dpi=600 );